# Семинар 8: Решающие деревья.

## Вступление
Сегодня мы познакомимся с новым типом моделей — решающими деревьями. На первый взгляд они имеют совершенно другую природу, чем линейные модели. Однако дальше в курсе мы с вами увидим, что в них больше общего, чем может показаться на первый взгляд. Сами по себе решающие деревья используются в машинном обучении относительно редко, однако очень распространены методы, основанные на их композиции: Random Forest, XGBoost, LightGBM. О них мы поговорим на дальнейших занятиях.

### План семинара
1. Повторяем теорию про решающие деревья
2. Линейные модели VS решающие деревья
3. Переобучение решающих деревьев
4. Неустойчивость решающих деревьев
5. Решение практической задачи при помощи решающих деревьев

## 1. Повторяем теорию про решающие деревья

Пройдемся по основным идеям с лекции. Решающее дерево состоит из вершин, в каждой из которой записан **предикат**: логическое выражение вида $[x_{ij} < t]$. При построении дерева обучающая выборка последовательно разбивается на подгруппы согласно предикатам в дереве. На каждом этапе объекты, приходящие в вершину дерева, разбиваются на две подгруппы: одна отправляется в левое поддерево, вторая — в правое. Выбирать предикаты (находить номер признака $j$ и значение порога $t$) мы будем исходя из **критерия информативности** разбиения. Наша цель — найти такой предикат, который наилучшим образом разобьёт множество объектов $R_m$ на два подмножества $R_l$ и $R_r$. Наилучшим — значит, что внутри группы $R_l$ объекты будут как можно более похожи друг на друга, внутри группы $R_r$ объекты тоже будут как можно более похожи друг на друга, а вот между группами различие будет настолько большим, насколько получится. Различия и сходства объектов для поиска оптимального разбиения обычно формулируются в терминах разнообразия (дисперсии, энтропии и т.д.) целевой переменной.

Давайте попробуем сформулировать всё это на математическом языке. Для фиксированной вершины обозначим множество пришедших в неё объектов $R_m$, множества полученных подгрупп $R_l$ и $R_r$, номер признака разбиения $j$, значение порога для разбиения $t$ и $H(R)$ — критерий неоднородности (impurity criterion). Тогда мы можем записать условие того, что мы хотим разбить объекты $R_m$ на две группы, внутри которых объекты будут наиболее однородными:

$$H(R_m) - \frac{|R_l|}{|R_m|}H(R_l) - \frac{|R_r|}{|R_m|}H(R_r) \to \max.$$

Разумеется, конкретный набор элементов в $R_l$ и $R_r$ зависит от предиката, который мы построили. Предикат, в свою очередь, однозначно определяется двумя числами: $j$ и $t$. Также заметим, что в записанном выше выражении мы не можем контролировать $H(R_m)$, потому что это просто функция от тех объектов, которые пришли в вершину, и она не зависит от предиката. Тогда мы можем переписать наше условие в чуть более простом виде:

$$Q(R_m, j, t) = \frac{|R_\ell|}{|R_m|}H(R_\ell) + \frac{|R_r|}{|R_m|}H(R_r) \to \min_{j, t}.$$

Остался последний неясный момент: что же такое $H(R)$ и как её вычислять? Ответ зависит от задачи, которую мы решаем.

#### Регрессия
1. MSE — дисперсия целевой переменной
$$H(R) = \frac{1}{|R|} \sum_{(x_i, y_i) \in R}{}(y_i - \hat{y})^2$$

2. MAE — отклонение от медианы
$$H(R) = \frac{1}{|R|} \sum_{(x_i, y_i) \in R}{}\frac{|y_i - MEDIAN(Y)|}{|R|}$$

#### Классификация
Обозначим $p_k$ долю объектов класса $k$ во множестве объектов $R$.
1. Энтропия
$$H(R) = -\sum_{k=1}^{K} p_k log(p_k)$$

2. Критерий Джини
$$H(R) = -\sum_{k=1}^{K} p_k log(p_k)$$

Давайте на практике убедимся, что энтропийный критерий и правда работает: для однородной выборки возвращает значение больше, чем для выборки с явной в одном из классов.

In [ ]:
import numpy as np

g1 = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
g2 = np.array([0.1, 0.6, 0.1, 0.1, 0.1])

# Как посчитать энтропию для двух выборок (отдельно)?
res1 = ... # Ваш код сюда :)
res2 = ... # Ваш код сюда :)
print(res1, res2)

### Ответьте на вопросы:
1.  Какие методы регуляризации деревьев вы знаете?
2.  Сильно ли изменится дерево от небольшого изменения данных?
3.  Как происходит предсказание с использованием дерева в задачах регрессии и классификации?
4.  Как бы вы оценивали важность признаков при использовании решающего дерева?

### Визуализация решающего дерева
![](https://drive.google.com/uc?id=13kmqLXa-3FBUJq0Q3XVOkz8TENpVTkqk)

## 2. Линейные модели VS решающие деревья

Раньше мы разбирали только линейные модели, которые имеют совсем другую природу по отношению к решающим деревьям. Можно ли сказать, что какой-то из этих двух типов моделей всегда лучше? Нет. В зависимости от пространственной структуры данных, один из них будет работать лучше:

- Линейная модель, если данные хорошо линейно разделимы
- Решающие деревья, если данные плохо линейно разделимы (присутствуют только кусочно-линейные или нелинейные зависимости)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlxtend.plotting import plot_decision_regions
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split

%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 6.5)

In [ ]:
np.random.seed(13)
n = 500
# Как создать вектор нулей с shape=(n, 2)?
X = ... # Ваш код сюда :)
X[:, 0] = np.linspace(-5, 5, 500)
X[:, 1] = X[:, 0] + 0.5 * np.random.normal(size=n)
# Как создать target (y: 1 и 0) для случая X[:, 1] > X[:, 0]?
y = ... # Ваш код сюда :)
plt.scatter(X[:, 0], X[:, 1], s=100, c=y, cmap="winter");

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=13
)

# Как создать / обучить линейную классификацию с random_state=13?
# Также сделайте предсказание для такой модели.
lr = ... # Ваш код сюда :)
... # Ваш код сюда :)
y_pred_lr = ... # Ваш код сюда :)

print(f"Linear model accuracy: {accuracy_score(y_pred_lr, y_test):.2f}")

In [ ]:
plot_decision_regions(X_test, y_test, lr);

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=13)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(f"Decision tree accuracy: {accuracy_score(y_pred_dt, y_test):.2f}")

In [ ]:
X_test.shape

In [ ]:
dt.get_depth()

In [ ]:
import sklearn
import matplotlib.pyplot

plt.figure(figsize=(17,10))
sklearn.tree.plot_tree(dt);

In [ ]:
plot_decision_regions(X_test, y_test, dt);

In [ ]:
np.random.seed(13)
X = np.random.randn(500, 2)
y = np.logical_xor(X[:, 0] > 0, X[:, 1] > 0).astype(int)
plt.scatter(X[:, 0], X[:, 1], s=100, c=y, cmap="winter");

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=13
)

lr = LogisticRegression(random_state=13)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print(f"Linear model accuracy: {accuracy_score(y_pred_lr, y_test):.2f}")

plot_decision_regions(X_test, y_test, lr);

In [ ]:
dt = DecisionTreeClassifier(random_state=13)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(f"Linear model accuracy: {accuracy_score(y_pred_dt, y_test):.2f}")

plot_decision_regions(X_test, y_test, dt);

## 3. Переобучение решающих деревьев
На лекции рассказывалось, что решающие деревья способны переобучиться под любую выборку, если их никак не регуляризовать: при большом количестве листьев для каждого объекта может выделиться своя область в признаковом пространстве. По сути дерево просто выучивает обучающую выборку, не выделяет никаких закономерностей среди данных. Давайте убедимся в этом эффекте на практике, сгенерировав два трудноразделимых множества объектов.

In [ ]:
np.random.seed(13)
n = 100
X = np.random.normal(size=(n, 2))
X[:50, :] += 0.25
X[50:, :] -= 0.25
y = np.array([1] * 50 + [0] * 50)
plt.scatter(X[:, 0], X[:, 1], s=100, c=y, cmap="winter");

Посмотрим, как влияют разные значения гиперпараметров решающего дерева на его структуру:

- `max_depth`: максимальная глубина дерева
- `min_samples_leaf`: минимальное число объектов в вершине дерева, необходимое для того, чтобы она стала листовой

In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))

for i, max_depth in enumerate([3, 5, None]):
    for j, min_samples_leaf in enumerate([15, 5, 1]):
        dt = DecisionTreeClassifier(
            max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=13
        )
        dt.fit(X, y)
        ax[i][j].set_title(
            "max_depth = {} | min_samples_leaf = {}".format(max_depth, min_samples_leaf)
        )
        ax[i][j].axis("off")
        plot_decision_regions(X, y, dt, ax=ax[i][j])

plt.show()

На любой выборке (исключая те, где есть объекты с одинаковыми значениями признаков, но разными ответами) можно получить нулевую ошибку с помощью максимально переобученного дерева:

In [ ]:
dt = DecisionTreeClassifier(max_depth=None, min_samples_leaf=1, random_state=13)
dt.fit(X, y)

print(f"Decision tree accuracy: {accuracy_score(y, dt.predict(X)):.2f}")

plot_decision_regions(X, y, dt);

 ## 4. Неустойчивость решающих деревьев

Посмотрим, как будет меняться структура дерева, если брать для обучения разные 90%-ые подвыборки исходной выборки.

In [ ]:
fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))

for i in range(3):
    for j in range(3):
        seed_idx = 3 * i + j
        np.random.seed(seed_idx)
        dt = DecisionTreeClassifier(random_state=13)
        idx_part = np.random.choice(len(X), replace=False, size=int(0.9 * len(X)))
        X_part, y_part = X[idx_part, :], y[idx_part]
        dt.fit(X_part, y_part)
        ax[i][j].set_title("sample #{}".format(seed_idx))
        ax[i][j].axis("off")
        plot_decision_regions(X_part, y_part, dt, ax=ax[i][j])

plt.show()

## 5. Решение практической задачи при помощи решающих деревьев

In [ ]:
california = fetch_california_housing()
california_X = pd.DataFrame(data=california.data, columns=california.feature_names)
california_Y = california.target
print(f"X shape: {california_X.shape}, Y shape: {california_Y.shape}")
california_X.head()

In [ ]:
plt.title("House price distribution")
plt.xlabel("price")
plt.ylabel("# samples")
plt.hist(california_Y, bins=20)
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    california_X, california_Y, test_size=0.25, random_state=13
)

dt = DecisionTreeRegressor(max_depth=3, random_state=13)
dt.fit(X_train, y_train)

plot_tree(dt, feature_names=california_X.columns, filled=True, rounded=True)
plt.show()

In [ ]:
mean_squared_error(y_test, dt.predict(X_test))

In [ ]:
max_depth_array = range(2, 20)
mse_array = []

for max_depth in max_depth_array:
    dt = DecisionTreeRegressor(max_depth=max_depth, random_state=13)
    dt.fit(X_train, y_train)
    mse_array.append(mean_squared_error(y_test, dt.predict(X_test)))

plt.plot(max_depth_array, mse_array)
plt.title("Dependence of MSE on max depth")
plt.xlabel("max depth")
plt.ylabel("MSE");

In [ ]:
pd.DataFrame({"max_depth": max_depth_array, "MSE": mse_array}).sort_values(
    by="MSE"
).reset_index(drop=True)

In [ ]:
min_samples_leaf_array = range(1, 20)
mse_array = []

for min_samples_leaf in min_samples_leaf_array:
    dt = DecisionTreeRegressor(
        max_depth=6, min_samples_leaf=min_samples_leaf, random_state=13
    )
    dt.fit(X_train, y_train)
    mse_array.append(mean_squared_error(y_test, dt.predict(X_test)))

plt.plot(min_samples_leaf_array, mse_array)
plt.title("Dependence of MSE on min samples leaf")
plt.xlabel("min samples leaf")
plt.ylabel("MSE");

In [ ]:
min_samples_split_array = range(2, 20)
mse_array = []

for min_samples_split in min_samples_split_array:
    dt = DecisionTreeRegressor(
        max_depth=6, min_samples_split=min_samples_split, random_state=13
    )
    dt.fit(X_train, y_train)
    mse_array.append(mean_squared_error(y_test, dt.predict(X_test)))

plt.plot(min_samples_split_array, mse_array)
plt.title("Dependence of MSE on min samples split")
plt.xlabel("min samples split")
plt.ylabel("MSE");

In [ ]:
dt = DecisionTreeRegressor(max_depth=6, random_state=13)
dt.fit(X_train, y_train)
plot_tree(dt, feature_names=california_X.columns, filled=True, rounded=True);

In [ ]:
mean_squared_error(y_test, dt.predict(X_test))

In [ ]:
dt.feature_importances_

In [ ]:
pd.DataFrame(
    {"feature": california_X.columns, "importance": dt.feature_importances_}
).sort_values(by="importance", ascending=False).reset_index(drop=True)

Влияет ли стандартизация (масштабирование) признаков на результат работы решающего дерева?

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X_train.head()

In [ ]:
sc = StandardScaler()
X_train_scaled = pd.DataFrame(
    sc.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    sc.transform(X_test), columns=X_test.columns, index=X_test.index
)
X_train_scaled.head()

In [ ]:
print("No scaling is applied\n")

for max_depth in [3, 6]:
    dt = DecisionTreeRegressor(max_depth=max_depth, random_state=13)
    dt.fit(X_train, y_train)
    print(
        f"MSE on test set for depth {max_depth}: {mean_squared_error(y_test, dt.predict(X_test)):.2f}"
    )

In [ ]:
print("Standard scaling is applied\n")

for max_depth in [3, 6]:
    dt = DecisionTreeRegressor(max_depth=max_depth, random_state=13)
    dt.fit(X_train_scaled, y_train)
    print(
        f"MSE on test set for depth {max_depth}: {mean_squared_error(y_test, dt.predict(X_test_scaled)):.2f}"
    )